## Using MCP in LlamaIndex

In [3]:
%pip install llama-index-tools-mcp

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\sobha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.workflow import Context
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.tools.mcp import McpToolSpec

llm = HuggingFaceInferenceAPI(model_name="meta-llama/Llama-3.2-3B-Instruct")

SYSTEM_PROMPT = "You are an AI assistant. Use the available tools to answer the user's questions."


async def get_agent(tool_spec: McpToolSpec) -> FunctionAgent:
    # fetch the tools exposed by the MCP server and wrap them in an agent
    tools = await tool_spec.to_tool_list_async()
    return FunctionAgent(
        name="Agent",
        description="An agent that can use the tools exposed by the MCP server.",
        tools=tools,
        llm=llm,
        system_prompt=SYSTEM_PROMPT,
    )

In [7]:
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

# We consider there is a mcp server running on 127.0.0.1:8000, or you can use the mcp client to connect to your own mcp server.
mcp_client = BasicMCPClient("http://127.0.0.1:8000/sse")
mcp_tool = McpToolSpec(client=mcp_client)

# get the agent
agent = await get_agent(mcp_tool)

# create the agent context
agent_context = Context(agent)